## NB For Testing / Recording Bitboard Implementation

Actual implementation is made in Cython file tablut.pyx

Board representation is 2 quad words in struct 'Bitboard'. First 63 bits (0-62) of Bitboard.low represents the 'bottom 7' ranks, first 18 bits (0-17) of Bitboard.high represents the 'top 2' ranks. I used this representation as it was the approach taken in the most sophisticated 9x9 open source board game implementation I readily found - Shogi program [Apery](https://github.com/HiraokaTakuya/apery_rust/blob/master/src/bitboard.rs). I believe this also affords us the elegant solution of merging the high and low boards with a simple bitwise OR function when creating a key for move generation.

### Checklist
__Groundwork__
- Decide on architecture and define it in a struct
- Write bit manipulation macros that work across both quad words
- Write basic board setting / clearing / printing functions

__Move Generation__
- Calculate moves 'on the fly'
- Pre-calculate blocker masks for all squares
- Generate keys from blocker masked board states
- Use Carry-Rippler trick to generate all possible blocker states for each square
- Use brute force trial and error to find magic numbers
- create a move table for index / move pairs

__Move Making__
- Create multiple Bitboards to track pieces
- Mask occupied Bitboard with move mask for legal moves

__Attacks__
- TBD (mask two over allies, map this to attacked squares, AND this with opponents for attacks?)

In [1]:
import tablut

In [2]:
board = tablut.create_start_position()
tablut.print_board(board)

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217272 28688
Bitboard (hex): 0x04223FF888402038 0x0000000000007010



In [3]:
tablut.print_blocker_bits()

14, 13, 13, 13, 13, 13, 13, 13, 14, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
13, 12, 12, 12, 12, 12, 12, 12, 13, 
14, 13, 13, 13, 13, 13, 13, 13, 14, 


In [4]:
blockers, moves = tablut.build_tables(80)

14
16384
9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 0 0
Bitboard (hex): 0x0000000000000000 0x0000000000000000

9 1 . . . . . . . . 
8 1 . . . . . . . . 
7 1 . . . . . . . . 
6 1 . . . . . . . . 
5 1 . . . . . . . . 
4 1 . . . . . . . . 
3 1 . . . . . . . . 
2 1 . . . . . . . . 
1 . 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: 18049651735528446 513
Bitboard (hex): 0x00402010080403FE 0x0000000000000201

----------------------------------
9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . 1 . 1 . . . . 
  a b c d e f g h i
Bitboard: 20 0
Bitboard (hex): 0x0000000000000014 0x0000000000000000

9 1 . . . . . . . . 
8 1 . . . . . . . . 
7 1 . . . . . . . . 
6 1 . . . . . . . . 
5 1 . . . . . . . .

In [5]:
2**14

16384

In [12]:
moves

[{'low': 18049651735528446, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527942, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527950, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527942, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527966, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527942, 'high': 513},
 {'low': 18049651735527936, 'high': 513},
 {'low': 18049651735527938, 'high': 513},
 {'low': 18049651735527936, 'high'

In [5]:
2**14

16384

In [2]:
tablut.check_subsets()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 0 0
Bitboard (hex): 0x0000000000000000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . 1 . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 16384 0
Bitboard (hex): 0x0000000000004000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . 1 . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 8388608 0
Bitboard (hex): 0x0000000000800000 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . 1 . . . 
2 . . . .

In [13]:
2 ** 12

4096

In [4]:
moves = tablut.find_moves_otf(start_pos, 77)
moves.print_board()

9 . . . . . . 1 1 1 
8 . . . . . 1 . . . 
7 . . . . . 1 . . . 
6 . . . . . 1 . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 577586652210266112 229408
Bitboard (hex): 0x0804000000000000 0x0000000000038020



In [2]:
tablut.generate_attack_masks()

BitboardStruct(low = 0x00402010080403FE, high = 0x0000000000000201),
BitboardStruct(low = 0x00804020100805FD, high = 0x0000000000000402),
BitboardStruct(low = 0x01008040201009FB, high = 0x0000000000000804),
BitboardStruct(low = 0x02010080402011F7, high = 0x0000000000001008),
BitboardStruct(low = 0x04020100804021EF, high = 0x0000000000002010),
BitboardStruct(low = 0x08040201008041DF, high = 0x0000000000004020),
BitboardStruct(low = 0x10080402010081BF, high = 0x0000000000008040),
BitboardStruct(low = 0x201008040201017F, high = 0x0000000000010080),
BitboardStruct(low = 0x40201008040200FF, high = 0x0000000000020100),
BitboardStruct(low = 0x004020100807FC01, high = 0x0000000000000201),
BitboardStruct(low = 0x00804020100BFA02, high = 0x0000000000000402),
BitboardStruct(low = 0x010080402013F604, high = 0x0000000000000804),
BitboardStruct(low = 0x020100804023EE08, high = 0x0000000000001008),
BitboardStruct(low = 0x040201008043DE10, high = 0x0000000000002010),
BitboardStruct(low = 0x08040201008

In [9]:
rank = 3
pos = 0

masked_rank = start_pos.get_rank_key(rank)
_tmp = tablut.Bitboard(masked_rank)
_tmp.print_board()
moves = tablut.get_moves(masked_rank, pos)
moves.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 . . . 1 . . . 1 
  a b c d e f g h i
Bitboard: 273 0
Bitboard (hex): 0x0000000000000111 0x0000000000000000

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . 1 1 1 . . . . . 
  a b c d e f g h i
Bitboard: 14 0
Bitboard (hex): 0x000000000000000E 0x0000000000000000



In [18]:
target_rank = 3
start_pos_move_key = start_pos.shift_rank_to_bottom(target_rank)
start_pos_move_key.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 . . . 1 . . . 1 
  a b c d e f g h i
Bitboard: 273 0
Bitboard (hex): 0x0000000000000111 0x0000000000000000



In [9]:
tmp = tablut.Bitboard()

for i in range(36, 45):
    tmp.set_bit(i)

tmp.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 1 1 1 1 1 1 1 1 1 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . . . . . . . . . 
  a b c d e f g h i
Bitboard: 35115652612096 0
Bitboard (hex): 0x00001FF000000000 0x0000000000000000



In [10]:
_tmp = tmp.shift_rank_to_bottom(4)
_tmp.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 1 1 1 1 1 1 1 1 1 
  a b c d e f g h i
Bitboard: 511 0
Bitboard (hex): 0x00000000000001FF 0x0000000000000000



In [25]:
piece_pos = 4
moves = tablut.get_moves(start_pos_move_key, piece_pos)
moves.print_board()

9 . . . . . . . . . 
8 . . . . . . . . . 
7 . . . . . . . . . 
6 . . . . . . . . . 
5 . . . . . . . . . 
4 . . . . . . . . . 
3 . . . . . . . . . 
2 . . . . . . . . . 
1 . 1 1 1 . 1 1 1 . 
  a b c d e f g h i
Bitboard: 238 0
Bitboard (hex): 0x00000000000000EE 0x0000000000000000



In [3]:
board.print_board()

9 . . . 1 1 1 . . . 
8 . . . . 1 . . . . 
7 . . . . 1 . . . . 
6 1 . . . 1 . . . 1 
5 1 1 1 1 1 1 1 1 1 
4 1 . . . 1 . . . 1 
3 . . . . 1 . . . . 
2 . . . . 1 . . . . 
1 . . . 1 1 1 . . . 
  a b c d e f g h i
Bitboard: 297870862030217272 57376
Bitboard (hex): 0x04223FF888402038 0x000000000000E020



In [4]:
class ULL64:
    def __init__(self, value=0):
        self.value = value & ((1 << 64) - 1)  # Ensure 64-bit

    def __and__(self, other):
        return ULL64(self.value & other.value)

    def __or__(self, other):
        return ULL64(self.value | other.value)

    def __xor__(self, other):
        return ULL64(self.value ^ other.value)

    def __lshift__(self, shift):
        return ULL64(self.value << shift)

    def __rshift__(self, shift):
        return ULL64(self.value >> shift)

    def __invert__(self):
        return ULL64(~self.value & ((1 << 64) - 1))

    def __str__(self):
        return f"{self.value:064b}"

    def __repr__(self):
        return f"ULL64({self.value})"

    def __sub__(self, other):
        return ULL64(self.value - other.value)

    def bit_length(self):
        return self.value.bit_length()

    def set_bit(self, index):
        if 0 <= index < 64:
            self.value |= (1 << index)
        else:
            raise ValueError("Bit index must be between 0 and 63")

    def clear_bit(self, index):
        if 0 <= index < 64:
            self.value &= ~(1 << index)
        else:
            raise ValueError("Bit index must be between 0 and 63")

    def is_bit_set(self, index):
        if 0 <= index < 64:
            return bool(self.value & (1 << index))
        else:
            raise ValueError("Bit index must be between 0 and 63")

class Bitboard:
    def __init__(self, low=0, high=0):
        self.low = ULL64(low)
        self.high = ULL64(high)

    def set_bit(self, index):
        if 0 <= index < 62:
            self.low.set_bit(index)
        elif 62 <= index < 81:
            self.high.set_bit(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")

    def clear_bit(self, index):
        if 0 <= index < 62:
            self.low.clear_bit(index)
        elif 62 <= index < 81:
            self.high.clear_bit(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")

    def is_bit_set(self, index):
        if 0 <= index < 62:
            return self.low.is_bit_set(index)
        elif 62 <= index < 81:
            return self.high.is_bit_set(index - 62)
        else:
            raise ValueError("Bit index must be between 0 and 80")


    def clear_all(self):
        self.low.value = 0
        self.high.value = 0

    def print_board(self):
        for rank in range(8, -1, -1):
            print(f"{rank+1} ", end="")
            for file in range(9):
                sq = rank * 9 + file
                if sq < 62:
                    print("1 " if self.low.is_bit_set(sq) else ". ", end="")
                else:
                    print("1 " if self.high.is_bit_set(sq - 62) else ". ", end="")
            print()
        print("  a b c d e f g h i")
        print(f"Bitboard: {self.low.value} {self.high.value}")
        print(f"Bitboard (hex): 0x{self.low.value:016X} 0x{self.high.value:016X}\n")


In [10]:
import time

In [11]:
def generate_moves(board):
    key_list = []
    value_list = []
    for i in range(9):
        moves = ULL64()
        if board.is_bit_set(i):
            for j in range(1, 8):
                if i + j < 9:
                    if not board.is_bit_set(i + j):
                        moves.set_bit(i + j)
                    else:
                        break
                else:
                    break
            for j in range(1, 8):
                if i - j >= 0:
                    if not board.is_bit_set(i - j):
                            moves.set_bit(i - j)
                    else:
                        break
                else:
                    break
            key_list.append((board.low.value, i))
            value_list.append(moves)
    return key_list, value_list

def generate_combinations():
    key_list = []
    value_list = []
    board = Bitboard()
    for i in range(512):
        print(f"{i+1}/512 Positions Added", end='\r')
        board.clear_all()  # Clear the board before setting new bits
        for j in range(9):
            if i & (1 << j):
                board.set_bit(j)
        keys, values = generate_moves(board)
        key_list += keys
        value_list += values
    return key_list, value_list

keys, values = generate_combinations()
print('\njoining keys and values...')
move_table = dict(zip(keys, values))
print('Move table created')


KeyboardInterrupt: 

In [183]:
len(move_table)

2304

In [ ]:
def generate_attack_table():


In [8]:
2^14 * 4 + 2**13 * 28 + 2**12 *49

430138